# 02 — Train and evaluate locally

Compare a naive baseline with logistic regression and a small random forest. Use the shared `trainer` package so Vertex training later runs the same code.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    precision_recall_curve,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

from trainer.data import clean_churn_frame, load_raw_csv, split_features_and_target
from trainer.model import build_pipeline
from trainer.task import evaluate

raw = load_raw_csv(Path("../data/raw/Telco-Customer-Churn.csv"))
clean = clean_churn_frame(raw)
X, y = split_features_and_target(clean)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
y_train.mean(), y_test.mean()

## Baseline

A dummy model that always predicts the majority class (no churn).

In [ ]:
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
dummy_pred = dummy.predict(X_test)
print(classification_report(y_test, dummy_pred, zero_division=0))

In [ ]:
results = {}
for name in ["logistic", "random_forest"]:
    pipe = build_pipeline(name)
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    results[name] = {"model": pipe, "proba": proba, "pred": pred, "metrics": evaluate(y_test, pred, proba)}

pd.DataFrame({k: v["metrics"] for k, v in results.items()}).T.round(3)

## Confusion matrix and ROC

Ask: is a missed churner (false negative) worse than a wasted retention offer (false positive)?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y_test, results["logistic"]["pred"], ax=axes[0], colorbar=False)
axes[0].set_title("Logistic @ 0.5")
fpr, tpr, _ = roc_curve(y_test, results["logistic"]["proba"])
axes[1].plot(fpr, tpr, label=f"AUC={results['logistic']['metrics']['roc_auc']:.3f}")
axes[1].plot([0, 1], [0, 1], linestyle="--", color="gray")
axes[1].set_title("ROC")
axes[1].set_xlabel("FPR")
axes[1].set_ylabel("TPR")
axes[1].legend()
plt.tight_layout()

## Thresholds

0.5 is not a business decision. If a retention call is cheap, lower the threshold to catch more churners.

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_test, results["logistic"]["proba"])
plt.figure(figsize=(6, 4))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall (logistic)")
plt.show()

In [ ]:
from trainer.task import train_and_export

metrics = train_and_export(
    Path("../data/raw/Telco-Customer-Churn.csv"),
    Path("../artifacts/model"),
    "logistic",
)
print(json.dumps(metrics, indent=2))